In [2]:
# ── TF 2.20 wheels (Kaggle only) ───────────────────────────────────────────
import os

KAGGLE = os.path.exists('/kaggle')
TRAIN_MODE = not KAGGLE   # False = inference / submission only

if KAGGLE:
    import subprocess
    subprocess.run(['pip', 'install', '-q', '--no-deps',
        '/kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl',
        '/kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorboard-2.20.0-py3-none-any.whl',
    ], check=True)
    # subprocess.run(['pip', 'install', 'tf2onnx', 'onnxruntime', 'onnxscript'], check=True)


os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [43]:
import os, random, logging, time, warnings
from contextlib import contextmanager
from pathlib import Path

# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# os.environ['CUDA_VISIBLE_DEVICES'] = ''   # Perch CPU model only

subprocess.run(['pip', 'install', 'tf2onnx', 'onnxruntime', 'onnxscript'], check=True)

import h5py
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import tf2onnx, onnx
import onnxruntime as ort
import onnxscript
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')
import torch
import torch.onnx
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

print('TF:', tf.__version__, '  PyTorch:', torch.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 7.9 MB/s eta 0:00:00
TF: 2.20.0   PyTorch: 2.10.0+cpu


## Config

In [44]:
# ── Paths ───────────────────────────────────────────────────────────────────
if KAGGLE:
    COMP_DIR    = Path('/kaggle/input/competitions/birdclef-2026')
    MODEL_DIR   = Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1')
    CKPT_DIR    = Path('/kaggle/working/checkpoints')
    HDF5_DIR    = Path('/kaggle/working/hdf5')
    OUTPUT_DIR  = Path('/kaggle/working')
    CKPT_PATH   = Path('/kaggle/input/models/mateomangialomini/perchv2-cpu/tensorflow2/default/1/perchv2_cpu.pt') 
else:
    ROOT        = (Path.home() / 'Documents/programming/birdclef+2026/birdclef-2026').resolve()
    COMP_DIR    = ROOT
    MODEL_DIR   = None   # kagglehub will download
    WORKING_DIR = ROOT / 'working'
    CKPT_DIR    = WORKING_DIR / 'checkpoints'
    HDF5_DIR    = WORKING_DIR / 'hdf5'
    OUTPUT_DIR  = WORKING_DIR / 'outputs'
    CKPT_PATH   = CKPT_DIR / 'fold0_best.pt'

TRAIN_AUDIO  = COMP_DIR / 'train_audio'
#TEST_SND_DIR = COMP_DIR / 'test_soundscapes'
TEST_SND_DIR = Path('/kaggle/input/datasets/mateomangialomini/test-dataset/')
TRAIN_SND    = COMP_DIR / 'train_soundscapes'
META_CSV     = COMP_DIR / 'train.csv'
SAMPLE_SUB   = COMP_DIR / 'sample_submission.csv'
HDF5_PATH    = HDF5_DIR / 'train_emb.h5'
OUT_PATH     = OUTPUT_DIR / 'submission.csv'

for d in [CKPT_DIR, HDF5_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if not TEST_SND_DIR.exists() or not any(TEST_SND_DIR.glob('*')):
    TEST_SND_DIR = TRAIN_SND
    print('test_soundscapes empty — using train_soundscapes')

# ── Audio ───────────────────────────────────────────────────────────────────
SAMPLE_RATE   = 32_000
CLIP_DURATION = 5
CLIP_SAMPLES  = SAMPLE_RATE * CLIP_DURATION

# ── Perch ───────────────────────────────────────────────────────────────────
PERCH_EMBED_DIM = 1536

# ── Training ────────────────────────────────────────────────────────────────
SEED            = 42
NUM_FOLDS       = 5
FOLD            = 0
TRAIN_EPOCHS    = 30
BATCH_SIZE      = 256
ACCUM_STEPS     = 1
LR              = 1e-3
WEIGHT_DECAY    = 1e-4
WARMUP_EPOCHS   = 2
LABEL_SMOOTHING = 0.05
MIXUP_ALPHA     = 0.4
NUM_WORKERS     = 4
PIN_MEMORY      = True
DROP_RATE       = 0.3

# ── Inference ───────────────────────────────────────────────────────────────
INFER_BATCH_SIZE = 64
AMP              = False
DEVICE           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cpu


## Perch Embedder

In [45]:
class PerchEmbedder:
    def __init__(self):
        if MODEL_DIR is not None and MODEL_DIR.exists():
            model_path = str(MODEL_DIR)
        else:
            import kagglehub
            model_path = kagglehub.model_download(
                'google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu/1'
            )
        print(f'Loading Perch from {model_path}')
        self.model    = tf.saved_model.load(model_path)
        self.infer_fn = self.model.signatures['serving_default']

        _test = tf.constant(np.zeros((1, 160_000), dtype=np.float32))
        _out  = self.infer_fn(inputs=_test)
        print('Output keys:', list(_out.keys()))
        self._emb_key = next(k for k in _out if 'embed' in k.lower())
        print(f'emb_key={self._emb_key}  shape={_out[self._emb_key].shape}')
        print('Perch loaded.')

    def embed(self, wave: np.ndarray) -> np.ndarray:
        """wave: (160_000,) → (1536,)"""
        x   = tf.constant(wave[np.newaxis], dtype=tf.float32)
        out = self.infer_fn(inputs=x)
        emb = out[self._emb_key].numpy().squeeze(0)   # (1536,) or (16,4,1536)
        if emb.ndim > 1:
            emb = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
        return emb

    def embed_batch(self, waves: np.ndarray) -> np.ndarray:
        """waves: (B, 160_000) → (B, 1536)"""
        x   = tf.constant(waves, dtype=tf.float32)
        out = self.infer_fn(inputs=x)
        emb = out[self._emb_key].numpy()              # (B, ..., 1536)
        if emb.ndim > 2:
            emb = emb.reshape(emb.shape[0], -1, emb.shape[-1]).mean(axis=1)
        return emb

## Utils

In [46]:
def get_logger(name='birdclef'):
    logger = logging.getLogger(name)
    if not logger.handlers:
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter('[%(asctime)s] %(levelname)s - %(message)s', '%H:%M:%S'))
        logger.addHandler(h)
        logger.setLevel(logging.INFO)
    return logger

logger = get_logger()


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


def build_label_map(meta_df):
    species = sorted(meta_df['primary_label'].unique().tolist())
    s2i = {s: i for i, s in enumerate(species)}
    i2s = {i: s for s, i in s2i.items()}
    return s2i, i2s


def encode_labels(primary, secondary, s2i):
    vec = np.zeros(len(s2i), dtype=np.float32)
    if primary in s2i:
        vec[s2i[primary]] = 1.0
    for lbl in (secondary or []):
        if lbl in s2i:
            vec[s2i[lbl]] = 1.0
    return vec


def competition_score(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average='macro'), keep


def mixup_data(x, y, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def save_checkpoint(state, path):
    torch.save(state, path)
    logger.info(f'Saved → {path}')


@contextmanager
def autocast_ctx():
    if AMP and torch.cuda.is_available():
        with torch.amp.autocast('cuda'):
            yield
    else:
        yield


def get_scaler():
    return torch.amp.GradScaler('cuda') if AMP and torch.cuda.is_available() else None

## Audio Processing

In [47]:
def load_wave(path):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        wave, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return wave.astype(np.float32)


def pad_or_trim(wave, length=CLIP_SAMPLES):
    if len(wave) < length:
        return np.pad(wave, (0, length - len(wave)))
    start = np.random.randint(0, len(wave) - length + 1)
    return wave[start : start + length]


def center_crop(wave, length=CLIP_SAMPLES):
    if len(wave) <= length:
        return np.pad(wave, (0, length - len(wave)))
    start = (len(wave) - length) // 2
    return wave[start : start + length]


def normalize_wave(wave):
    return wave / (np.abs(wave).max() + 1e-6)


def augment_wave(wave, p=0.5):
    if np.random.rand() < p:
        wave = wave + np.random.randn(*wave.shape).astype(np.float32) * 0.005
    if np.random.rand() < p:
        wave = np.roll(wave, int(np.random.uniform(-0.2, 0.2) * len(wave)))
    if np.random.rand() < p * 0.5:
        wave = wave * (10 ** (np.random.uniform(-6, 6) / 20))
    return wave


def chunk_wave(wave, window_samples=CLIP_SAMPLES):
    start, t = 0, CLIP_DURATION
    while start < len(wave):
        chunk = wave[start : start + window_samples]
        if len(chunk) < window_samples:
            chunk = np.pad(chunk, (0, window_samples - len(chunk)))
        yield chunk, t
        start += window_samples
        t     += CLIP_DURATION

## Precompute Embeddings (run once locally)

In [48]:
def precompute_embeddings(meta_df, embedder, out_path=HDF5_PATH, batch_size=32):
    already_done = set()
    if out_path.exists():
        with h5py.File(out_path, 'r') as f:
            already_done = set(f.keys())
        logger.info(f'Resuming — {len(already_done)} already cached')

    filenames = [r for r in meta_df['filename'].tolist() if r not in already_done]
    logger.info(f'Embedding {len(filenames)} clips → {out_path}')

    with h5py.File(out_path, 'a') as f:
        batch_waves, batch_keys = [], []

        def flush():
            if not batch_waves:
                return
            embs = embedder.embed_batch(np.stack(batch_waves))
            for key, emb in zip(batch_keys, embs):
                f.create_dataset(key, data=emb)
            batch_waves.clear(); batch_keys.clear()

        for fname in tqdm(filenames, desc='Embedding'):
            try:
                wave = normalize_wave(center_crop(load_wave(TRAIN_AUDIO / fname)))
                batch_waves.append(wave)
                batch_keys.append(fname)
                if len(batch_waves) >= batch_size:
                    flush()
            except Exception as e:
                logger.warning(f'Failed {fname}: {e}')
        flush()
    logger.info('Done.')


PRECOMPUTE_MODE = False
if PRECOMPUTE_MODE:
    embedder = PerchEmbedder()
    meta = pd.read_csv(META_CSV)
    precompute_embeddings(meta, embedder, batch_size=128)

## Dataset

In [49]:
class BirdDataset(Dataset):
    def __init__(self, df, s2i, mode='hdf5', augment=False, embedder=None):
        self.df      = df.reset_index(drop=True)
        self.s2i     = s2i
        self.mode    = mode
        self.augment = augment
        self.embedder = embedder
        self._h5 = None

    def _get_h5(self):
        if self._h5 is None:
            self._h5 = h5py.File(HDF5_PATH, 'r')
        return self._h5

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.mode == 'hdf5':
            emb = self._get_h5()[row['filename']][:]
            if emb.ndim > 1:
                emb = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
        else:
            wave = load_wave(TRAIN_AUDIO / row['filename'])
            wave = pad_or_trim(wave) if self.augment else center_crop(wave)
            if self.augment:
                wave = augment_wave(wave)
            wave = normalize_wave(wave)
            emb  = self.embedder.embed(wave)

        raw = row.get('secondary_labels', None)
        secondary = []
        if pd.notna(raw) and isinstance(raw, str) and raw.strip() not in ('', '[]'):
            secondary = [s.strip().strip("'\"") for s in raw.strip('[]').split(',') if s.strip()]

        return (
            torch.from_numpy(emb),
            torch.from_numpy(encode_labels(row['primary_label'], secondary, self.s2i))
        )


def get_fold_dfs(meta, fold=FOLD, n_folds=NUM_FOLDS, seed=SEED):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for f, (tr, val) in enumerate(skf.split(meta, meta['primary_label'])):
        if f == fold:
            return meta.iloc[tr].copy(), meta.iloc[val].copy()
    raise ValueError(f'Fold {fold} not found')


def get_loaders(meta, s2i, fold=FOLD, mode='hdf5', embedder=None):
    train_df, val_df = get_fold_dfs(meta, fold=fold)
    nw = NUM_WORKERS if mode == 'hdf5' else 0
    train_loader = DataLoader(
        BirdDataset(train_df, s2i, mode=mode, augment=True, embedder=embedder),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=nw,
        pin_memory=PIN_MEMORY, drop_last=True, persistent_workers=nw > 0,
    )
    val_loader = DataLoader(
        BirdDataset(val_df, s2i, mode=mode, augment=False, embedder=embedder),
        batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=nw,
        pin_memory=PIN_MEMORY, drop_last=False, persistent_workers=nw > 0,
    )
    return train_loader, val_loader

## Model

In [50]:
class PerchHead(nn.Module):
    def __init__(self, num_classes, embed_dim=PERCH_EMBED_DIM):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(DROP_RATE),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.head(x)


def build_model(num_classes):
    return PerchHead(num_classes)

## Training

In [51]:
class BCEWithLabelSmoothing(nn.Module):
    def __init__(self, pos_weight, smoothing=LABEL_SMOOTHING):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        return self.bce(logits, targets * (1 - self.smoothing) + self.smoothing / 2)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for step, (embs, labels) in enumerate(tqdm(loader, desc='  train', leave=False)):
        embs, labels = embs.to(device), labels.to(device)
        embs, y_a, y_b, lam = mixup_data(embs, labels)
        with autocast_ctx():
            loss = mixup_criterion(criterion, model(embs), y_a, y_b, lam) / ACCUM_STEPS
        (scaler.scale(loss) if scaler else loss).backward()
        if (step + 1) % ACCUM_STEPS == 0:
            if scaler:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            optimizer.zero_grad()
            scheduler.step()
        total_loss += loss.item() * ACCUM_STEPS
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    for embs, labels in tqdm(loader, desc='  valid', leave=False):
        embs, labels = embs.to(device), labels.to(device)
        with autocast_ctx():
            logits = model(embs)
            total_loss += criterion(logits, labels).item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    roc_auc, _ = competition_score(all_labels, all_preds)
    return total_loss / len(loader), roc_auc


if TRAIN_MODE:
    seed_everything()
    meta        = pd.read_csv(META_CSV)
    s2i, i2s    = build_label_map(meta)
    num_classes = len(s2i)

    train_loader, val_loader = get_loaders(meta, s2i, fold=FOLD, mode='hdf5')
    logger.info(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

    model     = build_model(num_classes).to(DEVICE)    
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = OneCycleLR(
        optimizer, max_lr=LR, total_steps=TRAIN_EPOCHS * len(train_loader),
        pct_start=WARMUP_EPOCHS / TRAIN_EPOCHS, anneal_strategy='cos',
    )

    # class-balanced pos_weight
    train_df, _ = get_fold_dfs(meta)
    label_counts = np.zeros(num_classes, dtype=np.float32)
    for _, row in train_df.iterrows():
        if row['primary_label'] in s2i:
            label_counts[s2i[row['primary_label']]] += 1
    pos_weight = torch.from_numpy(
        np.clip((len(train_df) - label_counts) / (label_counts + 1e-6), 1.0, 100.0)
    ).to(DEVICE)

    criterion = BCEWithLabelSmoothing(pos_weight=pos_weight)
    scaler    = get_scaler()
    best_score = 0.0

    for epoch in range(1, TRAIN_EPOCHS + 1):
        logger.info(f'Epoch {epoch}/{TRAIN_EPOCHS}')
        tr_loss = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, DEVICE)
        val_loss, roc_auc = validate(model, val_loader, criterion, DEVICE)
        logger.info(f'  tr={tr_loss:.4f}  val={val_loss:.4f}  roc_auc={roc_auc:.4f}  lr={scheduler.get_last_lr()[0]:.2e}')
        if roc_auc > best_score:
            best_score = roc_auc
            save_checkpoint({'epoch': epoch, 'model': model.state_dict(),
                             'optimizer': optimizer.state_dict(),
                             'best_score': best_score}, CKPT_DIR / f'fold{FOLD}_best.pt')
    logger.info(f'Best roc_auc: {best_score:.4f}')

## Inference

In [53]:
@torch.no_grad()
def predict_batch(embs, model, device):
    x = torch.from_numpy(np.stack(embs)).to(device)
    return torch.sigmoid(model(x)).cpu().numpy()


ss          = pd.read_csv(SAMPLE_SUB)
sub_species = [c for c in ss.columns if c != 'row_id']
meta        = pd.read_csv(META_CSV)
s2i, i2s    = build_label_map(meta)
num_classes = len(s2i)
train_species = [i2s[i] for i in range(num_classes)]
sub_col_idx   = {sp: i for i, sp in enumerate(sub_species)}
train_to_sub  = {ti: sub_col_idx[sp] for ti, sp in enumerate(train_species) if sp in sub_col_idx}
uniform_prior = 1.0 / len(sub_species)
logger.info(f'Mapped: {len(train_to_sub)}/{num_classes}  prior={uniform_prior:.4f}')

# Load embedder and trained head
embedder    = PerchEmbedder()
infer_model = build_model(num_classes)
ckpt        = torch.load(CKPT_PATH, map_location='cpu', weights_only=True)
infer_model.load_state_dict(ckpt['model'])
infer_model.to(DEVICE).eval()
logger.info(f'Loaded epoch={ckpt.get("epoch","?")}  best={ckpt.get("best_score",0):.4f}')

import torch.onnx

infer_model.eval()
dummy_input = torch.randn(1, PERCH_EMBED_DIM)  # (batch, 1536)

torch.onnx.export(
    infer_model,
    dummy_input,
    '/kaggle/working/mlp_head.onnx',
    input_names=['embedding'],
    output_names=['logits'],
    dynamic_axes={
        'embedding': {0: 'batch_size'},
        'logits': {0: 'batch_size'},
    },
    opset_version=18,
)
print('Exported MLP head to ONNX.')

ort_session = ort.InferenceSession('mlp_head.onnx', providers=['CPUExecutionProvider'])

def predict_batch_onnx(embs):
    x = np.stack(embs).astype(np.float32)
    logits = ort_session.run(['logits'], {'embedding': x})[0]
    return 1 / (1 + np.exp(-logits))  # sigmoid


# Discover test soundscapes from sample_submission row_ids
stems = ss['row_id'].str.rsplit('_', n=1).str[0].unique()
soundscape_paths = []
for stem in stems:
    for ext in ('.ogg', '.wav', '.flac'):
        p = TEST_SND_DIR / (stem + ext)
        if p.exists():
            soundscape_paths.append(p); break
if not soundscape_paths:
    soundscape_paths = sorted(TEST_SND_DIR.glob('*.ogg')) + sorted(TEST_SND_DIR.glob('*.wav'))
logger.info(f'Soundscapes: {len(soundscape_paths)}')

# Inference loop
import time
BATCH_FILES = 16
rows = []
t_load = t_embed = t_infer = 0.0

for start in tqdm(range(0, len(soundscape_paths), BATCH_FILES), desc='Inference'):
    batch_paths = soundscape_paths[start : start + BATCH_FILES]
    batch_n = len(batch_paths)

    t0 = time.perf_counter()
    x = np.zeros((batch_n * 12, CLIP_SAMPLES), dtype=np.float32)
    meta_batch = []
    for i, path in enumerate(batch_paths):
        wave, sr = sf.read(path, dtype='float32')
        if wave.ndim > 1:
            wave = wave.mean(axis=1)
        if len(wave) < 12 * CLIP_SAMPLES:
            wave = np.pad(wave, (0, 12 * CLIP_SAMPLES - len(wave)))
        else:
            wave = wave[:12 * CLIP_SAMPLES]
        x[i*12:(i+1)*12] = wave.reshape(12, CLIP_SAMPLES)
        meta_batch.extend([(path.stem, t) for t in range(5, 65, 5)])
    t_load += time.perf_counter() - t0

    t0 = time.perf_counter()
    outputs  = embedder.infer_fn(inputs=tf.convert_to_tensor(x))
    all_embs = outputs['embedding'].numpy()
    t_embed += time.perf_counter() - t0

    t0 = time.perf_counter()
    #all_probs = predict_batch(list(all_embs), infer_model, DEVICE)
    all_probs = predict_batch_onnx(list(all_embs))
    t_infer += time.perf_counter() - t0

    for (stem, end_t), probs in zip(meta_batch, all_probs):
        out = np.full(len(sub_species), uniform_prior, dtype=np.float32)
        for ti, si in train_to_sub.items():
            out[si] = probs[ti]
        rows.append({'row_id': f'{stem}_{end_t}', **dict(zip(sub_species, out.tolist()))})

    if start == 0:
        print(f'First batch — load={t_load:.2f}s  embed={t_embed:.2f}s  infer={t_infer:.2f}s')

print(f'Total — load={t_load:.2f}s  embed={t_embed:.2f}s  infer={t_infer:.2f}s')
        

# Save submission
if not rows:
    logger.warning('No rows — filling with uniform prior')
    sub_final = ss.copy()
    for sp in sub_species:
        sub_final[sp] = uniform_prior
else:
    sub_pred  = pd.DataFrame(rows)
    sub_final = ss[['row_id']].merge(sub_pred, on='row_id', how='left')
    for sp in sub_species:
        sub_final[sp] = sub_final[sp].fillna(uniform_prior)

sub_final.to_csv(OUT_PATH, index=False)
logger.info(f'Saved → {OUT_PATH}  shape={sub_final.shape}')
sub_final.head(3)

[15:49:09] INFO - Mapped: 206/206  prior=0.0043


Loading Perch from /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1


[15:49:16] INFO - Loaded epoch=4  best=0.9375


Output keys: ['embedding', 'spatial_embedding', 'spectrogram', 'label']
emb_key=embedding  shape=(1, 1536)
Perch loaded.


/tmp/ipykernel_55/3123200985.py:31: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


...

Inference:   9%|▉         | 1/11 [00:42<07:03, 42.35s/it]

First batch — load=0.28s  embed=42.05s  infer=0.00s


Inference: 100%|██████████| 11/11 [06:56<00:00, 37.82s/it]
[15:56:14] INFO - Saved → /kaggle/working/submission.csv  shape=(3, 235)


Total — load=3.15s  embed=412.76s  infer=0.02s


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


In [ ]:
# import os

# directory = "/kaggle/input"
# max_files = 20
# count = 0

# walker = os.walk(directory)

# while count < max_files:
#     try:
#         root, dirs, files = next(walker)
#     except StopIteration:
#         break
#     for file in files:
#         if count >= max_files:
#             break
#         print(os.path.join(root, file))
#         count += 1

# print(f"Found {count} files")

In [ ]:
# print('TEST_SND_DIR:', TEST_SND_DIR)
# print('exists:', TEST_SND_DIR.exists())
# print('files:', list(TEST_SND_DIR.iterdir())[:5])
# print('sample row_ids:', ss['row_id'].head(3).tolist())
# print('stems:', ss['row_id'].str.rsplit('_', n=1).str[0].unique()[:3])